# Role of the nodes according to Guimera cartography

This notebook applies the Guimera cartography framework to classify node roles in Venice's public transport networks.

It performs graph preprocessing (removing self-loops and isolating the largest connected component), executes the Louvain community detection algorithm followed by Guimera role assignment multiple times to account for stochasticity, and consolidates the results by computing consensus roles (by majority vote) and averaging the z-score and participation coefficient metrics across repetitions.

In [1]:
import sys
sys.path.append('../bin')

import guimera as gm
import networkx as nx

import pandas as pd

g_files = ["no_carnival_tourist","carnival_tourist","no_carnival_residents","carnival_residents"]

## Execution of the Guimera algorithm

In this section, it iterates through all network profiles, loads the preprocessed graphs (including a som preparation operations), and then runs the Louvain community detection followed by Guimera role assignment.

Iniytially, it is imperative to eliminate self-links that offer no substantive information, but rather generate substantial noise Subsequently, it is imperative to eliminate nodes that possess a degree of 0 or, as illustrated in the example, to preserve exclusively the fully connected component.

The Louvain and Guimera algorithms are applied 10 times to account for algorithmic randomness. For each repetition, it stores the community membership, z-score, participation coefficient (P), and assigned role for every node, finally saving all results to CSV files for further analysis.

In [2]:
for n_file in g_files:

    G = nx.read_graphml('../models/'+n_file+'.graphml')

    for i in G.nodes:
        try:
            G.remove_edge(i, i)
        except:
            #
            pass

    # While not strictly necessary, it is often considered prudent to retain components that are fully connected while removing those that are unconnected or isolated.
    #
    # Obtain the largest strongly connected component
    largest_scc = max(nx.strongly_connected_components(G), key=len)
    # Create subgraph with that component
    G = G.subgraph(largest_scc).copy()

    # We run the Louvain and Guimera methods multiple times to account for randomness.
    # The Guimera algorithm depends on the community detection done by the Louvain algorithm, which is stochastic.
    n_reps = 10

    # Prepare dicts for each dataframe: node as index, columns as repeat
    cm_dict = {}  # node: [cm_rep0, cm_rep1, ...]
    z_dict = {}   # node: [z_rep0, ...]
    p_dict = {}   # node: [p_rep0, ...]
    role_dict = {} # node: [role_rep0, ...]

    nodes = list(G.nodes)
    for node in nodes:
        cm_dict[node] = []
        z_dict[node] = []
        p_dict[node] = []
        role_dict[node] = []

    for rep in range(n_reps):
        # Run Louvain
        cm = nx.community.louvain_communities(G, seed=None, threshold=0.0000001)
        # Run Guimera
        dZ, dP = gm.guimera(G, cm, weight='weight')
        # Get roles
        dRol = gm.role(dZ, dP)
        for node in nodes:
            node_cm = next((i for i, comm in enumerate(cm) if node in comm), None)
            cm_dict[node].append(node_cm)
            z_dict[node].append(dZ.get(node, None))
            p_dict[node].append(dP.get(node, None))
            role_dict[node].append(dRol.get(node, None))

    # Create DataFrames
    df_cm = pd.DataFrame.from_dict(cm_dict, orient='index', columns=[f'repeat_{i}' for i in range(n_reps)])
    df_z = pd.DataFrame.from_dict(z_dict, orient='index', columns=[f'repeat_{i}' for i in range(n_reps)])
    df_p = pd.DataFrame.from_dict(p_dict, orient='index', columns=[f'repeat_{i}' for i in range(n_reps)])
    df_role = pd.DataFrame.from_dict(role_dict, orient='index', columns=[f'repeat_{i}' for i in range(n_reps)])

    # Save DataFrames
    df_cm.to_csv(f"../models/partial_results/{n_file}_guimera_repeated_cm.csv")
    df_z.to_csv(f"../models/partial_results/{n_file}_guimera_repeated_z.csv")
    df_p.to_csv(f"../models/partial_results/{n_file}_guimera_repeated_p.csv")
    df_role.to_csv(f"../models/partial_results/{n_file}_guimera_repeated_role.csv")

As an example of the data obtained, the print for the last profile analysed is included here.

In [3]:
# Count the number of repetitions of each role for each stop
role_counts = df_role.apply(lambda row: row.value_counts(), axis=1).fillna(0).astype(int)
role_counts.head(5)

,R1 Ultra-peripheral,R2 Peripheral,R3 Non-hub connectors,R6 Connector hubs,R7 Multi-community transfer hubs
Punta Sabbioni DX,0,10,0,0,0
"Lido (S.M.E.) ""B""",0,0,0,10,0
S. Elena DX,0,10,0,0,0
Giardini Biennale SX,0,0,10,0,0
Crea,0,0,10,0,0


## Result collection and summarization

This section consolidates the results from multiple repetitions of the Guimera algorithm. 

There are two variants:

1. It computes role distributions for each node, identifies the most frequently assigned (majority) role for each stop and uses this value as the final one (being the most consistently identified).

2. It calculates the average z-scores (`z`) and participation coefficients (`P`) across all repetitions and derives consensus roles based on the averaged metrics, 

This provides robust estimates that account for the stochastic nature of the community detection algorithm.

### Role consolidation (Majority Voting)

This is the one considered for future analysis

In [4]:
# For each node, create a list of roles with their counts (excluding roles with count 0)
def get_role_distribution(row):
    role_counts = row.value_counts()
    # Filter out roles with count 0
    role_counts = role_counts[role_counts > 0]
    # Convert to list of tuples (role, count)
    return list(role_counts.items())

In [5]:
for n_file in g_files:
    # Load the roles DataFrame from CSV
    df_role = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_role.csv", index_col=0)
    role_distribution = df_role.apply(get_role_distribution, axis=1)

    print(f"Role distribution for {n_file}:")
    print(role_distribution)

    role_distribution.to_csv(f"../models/partial_results/{n_file}_guimera_repeated_role_summarized.csv")

Role distribution for no_carnival_tourist:
Punta Sabbioni DX             [(R2 Peripheral, 6), (R3 Non-hub connectors, 4)]
Lido (S.M.E.) "B"                                    [(R6 Connector hubs, 10)]
S. Elena DX                  [(R3 Non-hub connectors, 7), (R6 Connector hub...
Giardini Biennale SX         [(R3 Non-hub connectors, 7), (R6 Connector hub...
Crea                                             [(R3 Non-hub connectors, 10)]
                                                   ...                        
Santa Maria Del Mare                                     [(R2 Peripheral, 10)]
Bacini - Arsenale Nord                           [(R3 Non-hub connectors, 10)]
S. Pietro di Castello                            [(R3 Non-hub connectors, 10)]
P.le Roma (Hotel S. Chiar                        [(R3 Non-hub connectors, 10)]
Piazzale Roma                [(R3 Non-hub connectors, 9), (R6 Connector hub...
Length: 61, dtype: object
Role distribution for carnival_tourist:
Punta Sabbioni DX     

In [6]:
import random

def get_majority_role(row):
    role_counts = row.value_counts()
    if role_counts.empty:
        return ""
    max_count = role_counts.max()
    majority_roles = role_counts[role_counts == max_count].index.tolist()
    return random.choice(majority_roles)

In [7]:
for n_file in g_files:
    # Load the roles DataFrame from CSV
    df_role = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_role.csv", index_col=0)
    majority_roles = df_role.apply(get_majority_role, axis=1)

    print(f"Majority roles for {n_file}:")
    print(majority_roles)

    # Save the consolidated majority roles to CSV
    majority_roles.to_csv(f"../models/partial_results/{n_file}_guimera_repeated_role_consolidated.csv")

Majority roles for no_carnival_tourist:
Punta Sabbioni DX                    R2 Peripheral
Lido (S.M.E.) "B"                R6 Connector hubs
S. Elena DX                  R3 Non-hub connectors
Giardini Biennale SX         R3 Non-hub connectors
Crea                         R3 Non-hub connectors
                                     ...          
Santa Maria Del Mare                 R2 Peripheral
Bacini - Arsenale Nord       R3 Non-hub connectors
S. Pietro di Castello        R3 Non-hub connectors
P.le Roma (Hotel S. Chiar    R3 Non-hub connectors
Piazzale Roma                R3 Non-hub connectors
Length: 61, dtype: object
Majority roles for carnival_tourist:
Punta Sabbioni DX            R3 Non-hub connectors
Bacini - Arsenale Nord       R3 Non-hub connectors
Lido (S.M.E.) "B"                R6 Connector hubs
S. Elena DX                  R3 Non-hub connectors
S. Pietro di Castello        R3 Non-hub connectors
Giardini Biennale SX         R3 Non-hub connectors
Crea                          

### Role consolidation (z and P averaging)

In [8]:
# Average z for each node/stop for all 4 files and save the results
for n_file in g_files:
    df_z = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_z.csv", index_col=0)
    df_z_mean = df_z.mean(axis=1)

    # Save the mean z values to a new CSV
    df_z_mean.to_csv(f"../models/partial_results/{n_file}_guimera_repeated_z_mean.csv", header=['z_mean'])
    print(f"Mean z values for {n_file}:")
    print(df_z_mean)

Mean z values for no_carnival_tourist:
Punta Sabbioni DX             0.118115
Lido (S.M.E.) "B"            13.232370
S. Elena DX                   2.581263
Giardini Biennale SX          2.509154
Crea                         -0.531209
                               ...    
Santa Maria Del Mare         -0.584657
Bacini - Arsenale Nord        0.142555
S. Pietro di Castello        -0.114462
P.le Roma (Hotel S. Chiar     1.771422
Piazzale Roma                 2.421656
Length: 61, dtype: float64
Mean z values for carnival_tourist:
Punta Sabbioni DX             1.047342
Bacini - Arsenale Nord        0.820081
Lido (S.M.E.) "B"            13.784029
S. Elena DX                   2.163429
S. Pietro di Castello        -0.218025
Giardini Biennale SX          2.334141
Crea                         -0.566524
Arsenale DX                   1.665471
S. Zaccaria (Pieta') "A"     18.406703
Rialto Mercato                1.535462
S. Marta                     -0.475027
Certosa                      -0.623224
C

In [9]:
# Average P for each node/stop for all 4 files and save the results
for n_file in g_files:
    df_p = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_p.csv", index_col=0)
    df_p_mean = df_p.mean(axis=1)
    # Save the mean P values to a new CSV
    df_p_mean.to_csv(f"../models/partial_results/{n_file}_guimera_repeated_p_mean.csv", header=['p_mean'])

    print(f"Mean P values for {n_file}:")
    print(df_p_mean)

Mean P values for no_carnival_tourist:
Punta Sabbioni DX            0.647206
Lido (S.M.E.) "B"            0.686907
S. Elena DX                  0.684344
Giardini Biennale SX         0.680647
Crea                         0.637964
                               ...   
Santa Maria Del Mare         0.522495
Bacini - Arsenale Nord       0.687468
S. Pietro di Castello        0.683029
P.le Roma (Hotel S. Chiar    0.658209
Piazzale Roma                0.653676
Length: 61, dtype: float64
Mean P values for carnival_tourist:
Punta Sabbioni DX            0.657883
Bacini - Arsenale Nord       0.692502
Lido (S.M.E.) "B"            0.706414
S. Elena DX                  0.708145
S. Pietro di Castello        0.701465
Giardini Biennale SX         0.709217
Crea                         0.510957
Arsenale DX                  0.717021
S. Zaccaria (Pieta') "A"     0.694902
Rialto Mercato               0.696213
S. Marta                     0.661659
Certosa                      0.630471
Celestia                

In [10]:
for n_file in g_files:
    # Load mean z and p values
    df_z = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_z_mean.csv", index_col=0)
    df_p = pd.read_csv(f"../models/partial_results/{n_file}_guimera_repeated_p_mean.csv", index_col=0)
    # Convert to dictionaries
    dZ = df_z['z_mean'].to_dict()
    dP = df_p['p_mean'].to_dict()
    # Calculate roles using mean z and p
    dRol = gm.role(dZ, dP)
    # Save roles to CSV
    pd.Series(dRol).to_csv(f"../models/partial_results/{n_file}_guimera_repeated_role_mean.csv", header=['role'])